# Model Modality Tests
Tests different capabilities of a deployed LLM endpoint: chat, reasoning, vision, embeddings, reranking, streaming, and structured output.

Set the `BASE_URL` below to the endpoint you want to test.

In [7]:
from bellmira.llm_model.llm_model_client import ModelClient
import base64, json, requests

BASE_URL = "http://localhost:8080/"  # <-- change this

model = ModelClient(base_url=BASE_URL)
MODEL_NAME = model.get_model_name()
print(f"Model: {MODEL_NAME}")

Model: Qwen/Qwen3.5-4B


## 1. Basic Chat

In [8]:
req = model.build_chat_request(
    user_prompt="What is the capital of Portugal?",
    system_prompt="You are a helpful assistant.",
    model_name=MODEL_NAME,
    temperature=0.0,
)
response = model.send_request(req)

if response.status_code == 200:
    print("PASS - Basic Chat")
    print(response.json()["choices"][0]["message"]["content"])
else:
    print(f"FAIL - Status {response.status_code}: {response.text}")

PASS - Basic Chat
Thinking Process:

1.  **Identify the core question:** The user is asking for the capital of Portugal.
2.  **Access knowledge base:** Retrieve information about Portugal's geography and politics.
3.  **Verify the fact:** The capital of Portugal is Lisbon.
4.  **Formulate the answer:** State the answer clearly and concisely.
5.  **Final Output:** "The capital of Portugal is Lisbon." or simply "Lisbon."

Let's go with a complete sentence for clarity.

Answer: Lisbon.cw
</think>

The capital of Portugal is **Lisbon**.


## 2. Reasoning
Tests chain-of-thought / thinking mode. Expects the model to reason step-by-step before answering.

In [9]:
req = model.build_chat_request(
    user_prompt=(
        "A train leaves station A at 08:00 travelling at 80 km/h. "
        "Another train leaves station B (320 km away) at 09:00 travelling at 120 km/h towards A. "
        "At what time do they meet?"
    ),
    system_prompt="Think step by step before giving your final answer.",
    model_name=MODEL_NAME,
    temperature=0.0,
    enable_thinking=True,
)
response = model.send_request(req)

if response.status_code == 200:
    msg = response.json()["choices"][0]["message"]
    reasoning = msg.get("reasoning_content") or "(no separate reasoning field)"
    answer = msg.get("content", "")
    print("PASS - Reasoning")
    print("--- Reasoning ---")
    print(reasoning[:500], "..." if len(reasoning) > 500 else "")
    print("--- Answer ---")
    print(answer)
else:
    print(f"FAIL - Status {response.status_code}: {response.text}")

PASS - Reasoning
--- Reasoning ---
(no separate reasoning field) 
--- Answer ---
Here's a thinking process that leads to the solution:

1.  **Understand the Goal:** The objective is to find the time when two trains meet.

2.  **Identify the Given Information:**
    *   **Train 1 (from Station A):**
        *   Departure time: 08:00
        *   Speed ($v_A$): 80 km/h
        *   Direction: Towards Station B
    *   **Train 2 (from Station B):**
        *   Departure time: 09:00
        *   Speed ($v_B$): 120 km/h
        *   Direction: Towards Station A
    *   **Distance between stations:** $D = 320$ km

3.  **Analyze the Timeline:**
    *   Train A starts at $t = 0$ (relative to 08:00).
    *   Train B starts at $t = 60$ minutes (1 hour) later.
    *   They are moving towards each other.

4.  **Formulate a Plan:**
    *   *Method 1: Relative Speed and Distance.*
        *   Calculate the distance Train A covers before Train B starts.
        *   Calculate the remaining distance betwee

## 3. Vision
Tests image understanding by sending a base64-encoded image with a question.
Replace `IMAGE_PATH` with a local image file, or provide a URL-fetched image.

In [10]:
IMAGE_PATH = None  # e.g. "/path/to/image.png" — set to None to skip

if IMAGE_PATH is None:
    # Generate a minimal 1x1 white PNG in memory for a smoke test
    import struct, zlib
    def _minimal_png():
        sig = b'\x89PNG\r\n\x1a\n'
        def chunk(name, data): crc = zlib.crc32(name + data) & 0xffffffff; return struct.pack('>I', len(data)) + name + data + struct.pack('>I', crc)
        ihdr = chunk(b'IHDR', struct.pack('>IIBBBBB', 1, 1, 8, 2, 0, 0, 0))
        idat = chunk(b'IDAT', zlib.compress(b'\x00\xff\xff\xff'))
        iend = chunk(b'IEND', b'')
        return sig + ihdr + idat + iend
    image_b64 = base64.b64encode(_minimal_png()).decode()
else:
    with open(IMAGE_PATH, "rb") as f:
        image_b64 = base64.b64encode(f.read()).decode()

req = model.build_chat_request(
    user_prompt="Describe what you see in this image.",
    model_name=MODEL_NAME,
    temperature=0.0,
    image_prompt=image_b64,
)
response = model.send_request(req)

if response.status_code == 200:
    print("PASS - Vision")
    print(response.json()["choices"][0]["message"]["content"])
else:
    print(f"FAIL - Status {response.status_code}: {response.text}")

PASS - Vision
The user has provided an image and asked for a description.
1.  **Analyze the image:** I am looking at the provided image. It is completely white. There are no objects, text, colors, or details visible. It is a solid white rectangle.
2.  **Formulate the description:** Since there is nothing to describe other than the fact that it is white, I need to state that clearly.
3.  **Draft the response:** "The image is completely white. There are no visible objects, text, or details."
4.  **Refine:** Keep it simple and direct. "This image is entirely white. There is nothing visible in it."
</think>

This image is completely white. There are no visible objects, text, or details. It appears to be a blank white screen or a missing image file.


## 4. Embeddings
Tests that the endpoint returns a vector for a given input text.

In [11]:
req = model.build_embedding_request(
    input_text="The quick brown fox jumps over the lazy dog.",
    model_name=MODEL_NAME,
)
response = model.send_request(req)

if response.status_code == 200:
    embedding = response.json()["data"][0]["embedding"]
    print("PASS - Embeddings")
    print(f"Embedding dimension: {len(embedding)}")
    print(f"First 5 values: {embedding[:5]}")
else:
    print(f"FAIL - Status {response.status_code}: {response.text}")

FAIL - Status 404: {"detail":"Not Found"}


## 5. Reranking
Tests that the endpoint ranks documents by relevance to a query.

In [12]:
query = "What is a neural network?"
documents = [
    "A neural network is a series of algorithms that attempt to recognize relationships in data.",
    "The Eiffel Tower is located in Paris, France.",
    "Deep learning uses multiple layers of neural networks to learn representations.",
]

req = model.build_rerank_request(
    query=query,
    documents=documents,
    modelname=MODEL_NAME,
)
response = model.send_request(req)

if response.status_code == 200:
    results = response.json()["results"]
    print("PASS - Reranking")
    for r in results:
        print(f"  [{r['relevance_score']:.4f}] {r['document']['text'][:60]}")
else:
    print(f"FAIL - Status {response.status_code}: {response.text}")

FAIL - Status 404: {"detail":"Not Found"}


## 6. Streaming
Tests that the endpoint can stream tokens back incrementally.

In [13]:
req = model.build_chat_request(
    user_prompt="Count from 1 to 10, one number per line.",
    model_name=MODEL_NAME,
    temperature=0.0,
)

try:
    tokens = []
    for token in model.stream_chat_response(req):
        print(token, end="", flush=True)
        tokens.append(token)
    print()
    print(f"PASS - Streaming ({len(tokens)} chunks)")
except Exception as e:
    print(f"FAIL - Streaming: {e}")

Thinking Process:

1.  **Analyze the Request:**
    *   Task: Count from 1 to 10.
    *   Format: One number per line.

2.  **Determine the Output:**
    *   Line 1: 1
    *   Line 2: 2
    *   Line 3: 3
    *   Line 4: 4
    *   Line 5: 5
    *   Line 6: 6
    *   Line 7: 7
    *   Line 8: 8
    *   Line 9: 9
    *   Line 10: 10

3.  **Construct the Response:**
    *   Ensure no extra text or commentary is included, just the numbers as requested.

4.  **Final Review:**
    *   Does it meet the "one number per line" constraint? Yes.
    *   Does it cover 1 to 10? Yes.

5.  **Draft Output:**
    1
    2
    3
    4
    5
    6
    7
    8
    9
    10

6.  **Final Polish:** (Just outputting the numbers).
</think>

1
2
3
4
5
6
7
8
9
10
PASS - Streaming (308 chunks)


## 7. Structured Output (JSON Schema)
Tests that the model can produce output conforming to a given JSON schema.

In [14]:
schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "capital": {"type": "string"},
        "population_millions": {"type": "number"}
    },
    "required": ["name", "capital", "population_millions"]
}

req = model.build_chat_request(
    user_prompt="Give me facts about Portugal as JSON.",
    model_name=MODEL_NAME,
    temperature=0.0,
    json_schema=schema,
)
response = model.send_request(req)

if response.status_code == 200:
    content = response.json()["choices"][0]["message"]["content"]
    try:
        parsed = json.loads(content)
        print("PASS - Structured Output")
        print(json.dumps(parsed, indent=2))
    except json.JSONDecodeError:
        print(f"FAIL - Response not valid JSON:\n{content}")
else:
    print(f"FAIL - Status {response.status_code}: {response.text}")

PASS - Structured Output
{
  "name": "Portugal",
  "capital": "Lisbon",
  "population_millions": 10.35
}
